# Questão 2 — Normalização de Produtos

**Objetivo:** Limpar e padronizar o arquivo `produtos_raw.csv` para que os dados estejam confiáveis e prontos para análises futuras.

As três tarefas desta questão são:

1. **Parte 1:** Padronizar os nomes das categorias → `eletrônicos`, `propulsão`, `ancoragem`
2. **Parte 2:** Converter o campo `price` para tipo numérico
3. **Parte 3:** Remover linhas duplicadas

---

## 0. Importações e carregamento dos dados

In [16]:
import pandas as pd
import re
import unicodedata

df = pd.read_csv('../data/raw/produtos_raw.csv')

print(f'Shape original: {df.shape}')
print(f'Colunas: {df.columns.tolist()}')
print(f'\nTipos de dados:')
print(df.dtypes)
print(f'\nPrimeiras linhas:')
df.head()

Shape original: (157, 4)
Colunas: ['name', 'price', 'code', 'actual_category']

Tipos de dados:
name                 str
price                str
code               int64
actual_category      str
dtype: object

Primeiras linhas:


,name,price,code,actual_category
0,Transponder AIS Maré Magnum,R$ 33122.52,1,ELETRONICOS
1,Transponder Furuno Marlin,R$ 13998.15,2,ELETRONICOS
2,Radar Furuno Pulse Leviathan,R$ 9024.19,3,E L E T R Ô N I C O S
3,Rádio AIS Hydro Tidal Zen,R$ 3381.88,4,Eletrunicos
4,Piloto Automático Furuno Storm,R$ 23669.01,5,Eletronicoz


---
## Parte 1 — Padronização das Categorias

### Diagnóstico

O campo `actual_category` possui **39 variações distintas** para apenas 3 categorias reais. Os tipos de problema encontrados são:

| Tipo de problema | Exemplos |
|---|---|
| Capitalização inconsistente | `ELETRONICOS`, `eLeTrÔnIcOs`, `EletrônicoS` |
| Letras separadas por espaço | `E L E T R Ô N I C O S`, `P R O P U L S Ã O` |
| Erros de digitação | `Eletronicoz`, `Eletrunicos`, `Propulssão`, `Propução`, `Encoragem` |
| Abreviações | `Prop` (para propulsão) |

### Estratégia

1. Remover espaços entre letras isoladas (ex: `E L E T R` → `ELETR`)
2. Converter para minúsculas e remover acentos para comparação uniforme
3. Mapear para a categoria correta usando o **prefixo radical** de cada palavra — `elet` para eletrônicos, `prop` para propulsão, `anc`/`enc` para ancoragem

In [17]:
# Inspecionar todas as variações existentes antes de normalizar
print(f'Categorias únicas antes da normalização ({df["actual_category"].nunique()} variações):\n')
for cat in sorted(df['actual_category'].unique()):
    print(f'  "{cat}"')

Categorias únicas antes da normalização (39 variações):

  "A N C O R A G E M"
  "ANCORAGEM"
  "AnCoRaGeM"
  "AncorageM"
  "Ancoragem"
  "Ancoragen"
  "Ancoraguem"
  "AncorajeM"
  "Ancorajem"
  "Ancorajen"
  "Ancorajm"
  "E L E T R Ô N I C O S"
  "ELETRONICOS"
  "ELEtRÔNICOS"
  "Eletronicos"
  "Eletronicoz"
  "Eletroniscos"
  "Eletrunicos"
  "EletrônicoS"
  "Eletrônicos"
  "Encoragem"
  "Encoragi"
  "P R O P U L S Ã O"
  "PROPULSAO"
  "PrOpUlSãO"
  "Prop"
  "Propulsam"
  "Propulssão"
  "Propulçao"
  "Propulção"
  "Propução"
  "aNcOrAgEm"
  "ancoragem"
  "eLeTrÔnIcOs"
  "eletronicos"
  "eletrônicos"
  "pRoPuLsÃo"
  "propulsao"
  "propulsão"


In [18]:
def remover_acentos(texto):
    """Remove acentos de uma string para facilitar comparação."""
    return unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('ascii')


def normalizar_categoria(valor):
    """
    Normaliza qualquer variação de categoria para um dos três valores válidos:
    'eletrônicos', 'propulsão' ou 'ancoragem'.

    Passos:
      1. Remove espaços entre letras únicas separadas por espaço
         (ex: 'E L E T R Ô N I C O S' → 'ELETRÔNICOS')
      2. Converte para minúsculas e remove acentos para comparação
      3. Identifica a categoria pelo prefixo radical
    """
    # Passo 1: colapsar letras isoladas separadas por espaço
    sem_espacos = re.sub(r'(?<=\S) (?=\S)', '', valor).strip()

    # Passo 2: minúscula sem acento
    comparavel = remover_acentos(sem_espacos.lower())

    # Passo 3: mapear pelo radical
    if comparavel.startswith('elet'):
        return 'eletrônicos'
    elif comparavel.startswith('prop'):
        return 'propulsão'
    elif comparavel.startswith('anc') or comparavel.startswith('enc'):
        return 'ancoragem'
    else:
        return 'desconhecido'


df['category'] = df['actual_category'].apply(normalizar_categoria)

print('Categorias após normalização:')
print(df['category'].value_counts())
print(f'\nValores não mapeados: {(df["category"] == "desconhecido").sum()}')

Categorias após normalização:
category
propulsão      53
ancoragem      53
eletrônicos    51
Name: count, dtype: int64

Valores não mapeados: 0


In [19]:
# Validação: confirmar que cada valor original foi mapeado corretamente
validacao = (
    df[['actual_category', 'category']]
    .drop_duplicates()
    .sort_values(['category', 'actual_category'])
    .reset_index(drop=True)
)
print('Mapeamento completo (original → normalizado):')
print(validacao.to_string(index=False))

Mapeamento completo (original → normalizado):
      actual_category    category
    A N C O R A G E M   ancoragem
            ANCORAGEM   ancoragem
            AnCoRaGeM   ancoragem
            AncorageM   ancoragem
            Ancoragem   ancoragem
            Ancoragen   ancoragem
           Ancoraguem   ancoragem
            AncorajeM   ancoragem
            Ancorajem   ancoragem
            Ancorajen   ancoragem
             Ancorajm   ancoragem
            Encoragem   ancoragem
             Encoragi   ancoragem
            aNcOrAgEm   ancoragem
            ancoragem   ancoragem
E L E T R Ô N I C O S eletrônicos
          ELETRONICOS eletrônicos
          ELEtRÔNICOS eletrônicos
          Eletronicos eletrônicos
          Eletronicoz eletrônicos
         Eletroniscos eletrônicos
          Eletrunicos eletrônicos
          EletrônicoS eletrônicos
          Eletrônicos eletrônicos
          eLeTrÔnIcOs eletrônicos
          eletronicos eletrônicos
          eletrônicos eletrônicos
  

---
## Parte 2 — Conversão do Campo `price` para Numérico

### Diagnóstico

O campo `price` está armazenado como **string** no formato `R$ 9024.19`, com prefixo `R$ ` que impede operações matemáticas diretamente.

### Estratégia

1. Remover o prefixo `R$ `
2. Garantir que o separador decimal é ponto
3. Converter para `float`

In [20]:
# Antes: inspecionar o tipo e valores brutos
print(f'Tipo original de price: {df["price"].dtype}')
print(f'\nExemplos de valores brutos:')
print(df['price'].head(10).tolist())

Tipo original de price: str

Exemplos de valores brutos:
['R$ 33122.52', 'R$ 13998.15', 'R$ 9024.19', 'R$ 3381.88', 'R$ 23669.01', 'R$ 11820.21', 'R$ 19518.77', 'R$ 4984.15', 'R$ 39705.5', 'R$ 32033.04']


In [21]:
def converter_preco(valor):
    """
    Converte string de preço no formato 'R$ 9024.19' para float.
    Trata também o formato brasileiro 'R$ 9.024,19' caso apareça.
    """
    limpo = valor.replace('R$ ', '').strip()

    # Formato brasileiro com vírgula decimal: remover ponto de milhar e trocar vírgula
    if ',' in limpo:
        limpo = limpo.replace('.', '').replace(',', '.')

    return float(limpo)


df['price'] = df['price'].apply(converter_preco)

print(f'Tipo após conversão: {df["price"].dtype}')
print(f'\nEstatísticas descritivas de price (em R$):')
print(df['price'].describe().apply(lambda x: f'R$ {x:,.2f}'))

Tipo após conversão: float64

Estatísticas descritivas de price (em R$):
count        R$ 157.00
mean      R$ 35,194.62
std       R$ 42,183.18
min          R$ 309.54
25%        R$ 3,769.93
50%       R$ 13,704.10
75%       R$ 51,634.04
max      R$ 148,198.23
Name: price, dtype: str


A coluna de preços foi convertida de string para formato numérico, tratando tanto o padrão internacional quanto o formato brasileiro (com separador de milhar e vírgula decimal).

Após a conversão, foi possível calcular estatísticas descritivas, que indicam alta variabilidade nos preços, sugerindo a presença de produtos com valores significativamente elevados em relação à média.

---
## Parte 3 — Remoção de Duplicatas

### Diagnóstico

O dataset original tem 157 linhas para 150 produtos únicos. As 7 linhas extras são duplicatas de `code` geradas por variações no campo `actual_category` — o mesmo produto foi inserido múltiplas vezes com categorias grafadas de formas diferentes.

| Code | Produto | Linhas duplicadas |
|---|---|---|
| 37 | GPS Lowrance Evo Storm Drift | 2 |
| 62 | Motor Diesel Yanmar Velocity 37HP | 4 |
| 127 | Cabo de Nylon Delta Velocity Core Mako | 2 |
| 145 | Boia de Arqueamento Delta Nexus | 3 |

### Estratégia

Como `name` e `price` são idênticos em todas as linhas repetidas, basta manter a primeira ocorrência de cada `code` e descartar as demais. A coluna original `actual_category` também é removida, pois já foi substituída pela coluna limpa `category`.

In [22]:
# Confirmar os produtos duplicados antes da remoção
mask_dup = df.duplicated('code', keep=False)
duplicados = df[mask_dup][['code', 'name', 'actual_category']].sort_values('code')

print(f'Linhas com código duplicado: {len(duplicados)}')
print()
print(duplicados.to_string(index=False))

Linhas com código duplicado: 11

 code                                   name       actual_category
   37           GPS Lowrance Evo Storm Drift E L E T R Ô N I C O S
   37           GPS Lowrance Evo Storm Drift           ELEtRÔNICOS
   62      Motor Diesel Yanmar Velocity 37HP     P R O P U L S Ã O
   62      Motor Diesel Yanmar Velocity 37HP              Propução
   62      Motor Diesel Yanmar Velocity 37HP             propulsão
   62      Motor Diesel Yanmar Velocity 37HP                  Prop
  127 Cabo de Nylon Delta Velocity Core Mako             Encoragem
  127 Cabo de Nylon Delta Velocity Core Mako              Encoragi
  145        Boia de Arqueamento Delta Nexus             AncorageM
  145        Boia de Arqueamento Delta Nexus             AncorajeM
  145        Boia de Arqueamento Delta Nexus             Ancoragen


In [23]:
df_limpo = (
    df
    .drop(columns=['actual_category'])        # coluna suja, substituída por 'category'
    .drop_duplicates(subset='code', keep='first')
    .reset_index(drop=True)
)

print(f'Shape antes:  {df.shape}')
print(f'Shape depois: {df_limpo.shape}')
print(f'Linhas removidas: {len(df) - len(df_limpo)}')

Shape antes:  (157, 5)
Shape depois: (150, 4)
Linhas removidas: 7


---
## Resultado Final

Validação completa do dataset limpo e exportação.

In [24]:
# Reordenar colunas para melhor legibilidade
df_limpo = df_limpo[['code', 'name', 'category', 'price']]

print('=== DATASET FINAL ===')
print(f'Shape: {df_limpo.shape}')

print(f'\nTipos de dados:')
print(df_limpo.dtypes)

print(f'\nValores nulos por coluna:')
print(df_limpo.isnull().sum())

print(f'\nDistribuição por categoria:')
print(df_limpo['category'].value_counts())

print(f'\nCódigos duplicados restantes: {df_limpo.duplicated("code").sum()}')

=== DATASET FINAL ===
Shape: (150, 4)

Tipos de dados:
code          int64
name            str
category        str
price       float64
dtype: object

Valores nulos por coluna:
code        0
name        0
category    0
price       0
dtype: int64

Distribuição por categoria:
category
eletrônicos    50
propulsão      50
ancoragem      50
Name: count, dtype: int64

Códigos duplicados restantes: 0


In [25]:
print('Amostra do dataset limpo (primeiras 10 linhas):')
df_limpo.head(10)

Amostra do dataset limpo (primeiras 10 linhas):


,code,name,category,price
0,1,Transponder AIS Maré Magnum,eletrônicos,33122.52
1,2,Transponder Furuno Marlin,eletrônicos,13998.15
2,3,Radar Furuno Pulse Leviathan,eletrônicos,9024.19
3,4,Rádio AIS Hydro Tidal Zen,eletrônicos,3381.88
4,5,Piloto Automático Furuno Storm,eletrônicos,23669.01
5,6,Transponder AIS Vector,eletrônicos,11820.21
6,7,Radar AIS Zen,eletrônicos,19518.77
7,8,GPS AIS Zen,eletrônicos,4984.15
8,9,Transponder AIS Titan Pulse,eletrônicos,39705.50
9,10,Piloto Automático Simrad Titan Flux Magnum,eletrônicos,32033.04


In [26]:
df_limpo.to_csv('../data/processed/produtos_clean.csv', index=False)

print('Arquivo produtos_clean.csv salvo com sucesso.')
print()
print('Resumo das transformações aplicadas:')
print('  Parte 1 — Categorias normalizadas: 39 variações → 3 categorias limpas')
print('  Parte 2 — Preço convertido: string "R$ X" → float')
print('  Parte 3 — Duplicatas removidas: 157 linhas → 150 produtos únicos')

Arquivo produtos_clean.csv salvo com sucesso.

Resumo das transformações aplicadas:
  Parte 1 — Categorias normalizadas: 39 variações → 3 categorias limpas
  Parte 2 — Preço convertido: string "R$ X" → float
  Parte 3 — Duplicatas removidas: 157 linhas → 150 produtos únicos


# Questão 2.2 — Validação: Produtos Duplicados Removidos

**Pergunta:** Quantos produtos duplicados foram removidos?

---

In [27]:
import pandas as pd
import unicodedata

df = pd.read_csv('../data/raw/produtos_raw.csv')

print(f'Linhas no arquivo original: {len(df)}')

Linhas no arquivo original: 157


## Identificando as duplicatas

Antes de remover, foi inspecionado quais códigos aparecem mais de uma vez e quantas vezes cada um se repete.

In [28]:
duplicatas = df[df.duplicated('code', keep=False)].sort_values('code')

print(f'Linhas envolvidas em duplicação: {len(duplicatas)}')
print()
print(duplicatas[['code', 'name', 'price', 'actual_category']].to_string(index=False))

Linhas envolvidas em duplicação: 11

 code                                   name        price       actual_category
   37           GPS Lowrance Evo Storm Drift   R$ 6067.71 E L E T R Ô N I C O S
   37           GPS Lowrance Evo Storm Drift   R$ 6067.71           ELEtRÔNICOS
   62      Motor Diesel Yanmar Velocity 37HP R$ 102221.97     P R O P U L S Ã O
   62      Motor Diesel Yanmar Velocity 37HP R$ 102221.97              Propução
   62      Motor Diesel Yanmar Velocity 37HP R$ 102221.97             propulsão
   62      Motor Diesel Yanmar Velocity 37HP R$ 102221.97                  Prop
  127 Cabo de Nylon Delta Velocity Core Mako   R$ 1549.35             Encoragem
  127 Cabo de Nylon Delta Velocity Core Mako   R$ 1549.35              Encoragi
  145        Boia de Arqueamento Delta Nexus   R$ 4349.86             AncorageM
  145        Boia de Arqueamento Delta Nexus   R$ 4349.86             AncorajeM
  145        Boia de Arqueamento Delta Nexus   R$ 4349.86             Ancoragen


## Contando as linhas extras por código

Para cada código duplicado, a quantidade de linhas **extras** é o total de ocorrências menos 1 (a que será mantida).

In [29]:
contagem = df['code'].value_counts()
codigos_duplicados = contagem[contagem > 1]

print('Código | Produto                                  | Ocorrências | Linhas extras')
print('-' * 80)
total_extras = 0
for code, count in codigos_duplicados.items():
    nome = df[df['code'] == code]['name'].iloc[0]
    extras = count - 1
    total_extras += extras
    print(f'{code:>6} | {nome:<40} | {count:>11} | {extras:>13}')

print('-' * 80)
print(f'{'TOTAL':>72} | {total_extras:>13}')

Código | Produto                                  | Ocorrências | Linhas extras
--------------------------------------------------------------------------------
    62 | Motor Diesel Yanmar Velocity 37HP        |           4 |             3
   145 | Boia de Arqueamento Delta Nexus          |           3 |             2
    37 | GPS Lowrance Evo Storm Drift             |           2 |             1
   127 | Cabo de Nylon Delta Velocity Core Mako   |           2 |             1
--------------------------------------------------------------------------------
                                                                   TOTAL |             7


## Removendo as duplicatas e confirmando o resultado

In [30]:
linhas_antes = len(df)
df_clean = df.drop_duplicates(subset='code', keep='first').reset_index(drop=True)
linhas_depois = len(df_clean)
removidas = linhas_antes - linhas_depois

print(f'Linhas antes:  {linhas_antes}')
print(f'Linhas depois: {linhas_depois}')
print()
print(f'Produtos duplicados removidos: {removidas}')

Linhas antes:  157
Linhas depois: 150

Produtos duplicados removidos: 7
